
# COGS 108 - Final Project: Beyond the Pink Tax — Exploring Gender Differences in Clothing Prices

## Permissions

* [  ] YES - make available
* [X] NO - keep private


## Abstract

We did this project because we wanted to see whether the idea of a “pink tax” actually shows up in clothing prices. People often say women’s products cost more than men’s, so we wanted to test that with data instead of just assuming it was true. It felt like a good topic because it connects something really common, shopping, to a bigger issue about gender and fairness in pricing.

To do that, we used two different datasets and compared prices by gender in a few ways. One dataset let us compare men’s and women’s clothing directly, while the other used buyer gender to look at spending patterns. After analyzing the data, we found that the results were mixed and not strong enough to clearly prove a pink tax in clothing. In some cases, men’s clothing was actually more expensive, while in other cases women seemed to spend a little more on average, but the pattern was not consistent. Overall, we concluded that clothing prices seem to be influenced more by things like product type, season, and data structure than by gender alone.


## Authors

- **Yilin Cai**: Background research, Methodology, Software, Visualization, Writing – review & editing  
- **Nicole Liu**: Conceptualization, Background research, Data curation, Writing – original draft  
- **Xuanye Wang**: Data curation, Experimental investigation, Software  
- **Lianshi Deng**: Analysis, Background research, Conceptualization  
- **Jiangxi Fu**: Data curation, Project administration, Writing – original draft



## Research Question

Is there a statistically significant difference in the unit price of clothing items associated with women compared to men, and does that pattern still appear when gender is measured in two different ways: by **product label** and by **buyer gender**?



## Background and Prior Work

The idea of a “pink tax” usually refers to products marketed toward women costing more than similar products marketed toward men. Clothing is a useful category for studying this because apparel is often clearly gendered, is purchased repeatedly, and has been mentioned often in public discussions about gender-based pricing. At the same time, prior commentary has pointed out that these claims can be hard to evaluate because product comparisons are not always truly equivalent.

Our project builds on that discussion by comparing two separate datasets with different structures. One dataset lets us infer the **intended gender of the product** from item descriptions, which is closer to a direct test of gendered product pricing. The second dataset only includes **buyer gender**, which is a weaker proxy, but it still helps us check whether the broader pattern holds in a different retail setting. Looking at both sources lets us test whether the conclusion is robust or whether it depends heavily on how gender is defined in the data.



## Hypothesis

We hypothesized that female-associated clothing would have a higher unit price than male-associated clothing. If a pink tax is present in apparel, we would expect women’s products—or purchases associated with women—to show consistently higher prices across both datasets.



## Data

### Data overview

We used two Kaggle datasets:

**Dataset 1: Online Shopping / Product-Label Clothing Data**  
This dataset contains transaction-level retail records from an online store. We used product descriptions to infer whether an item was marketed toward men or women, then classified the relevant products into subcategories such as **T-Shirts & Shirts** and **Outerwear**. This dataset is the stronger test for our research question because the gender variable is tied to the product itself.

**Dataset 2: Retail Clothing Transactions / Buyer-Gender Proxy**  
This dataset contains retail transactions with variables such as age, gender, quantity, and price per unit. Unlike Dataset 1, gender here refers to the **customer**, not the product. That means it cannot directly test whether women’s clothing is priced higher, but it can still show whether spending patterns differ by gender in a separate shopping context.

Because the wrangling code for these datasets was fairly long, we do not reproduce all of it here. To see the detailed cleaning and feature engineering steps, see `02-EDACheckpoint.ipynb`. From this point forward, we simply load the processed data used for the final analysis.


In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
from scipy.stats import mannwhitneyu
from pathlib import Path

sns.set_theme(style='whitegrid')

_candidates = [Path.cwd(), Path.cwd().parent, Path('/content'), Path('/mnt/data')]
ROOT = next((p for p in _candidates if (p / 'data').exists()), Path.cwd())
PROCESSED = ROOT / 'data' / '02-processed'

print('ROOT =', ROOT)
print('PROCESSED =', PROCESSED)

# Load processed data created during the checkpoint phase
# See 02-EDACheckpoint.ipynb for full wrangling details.
d1 = pd.read_csv(PROCESSED / 'yilin_d1_online_clothing_clean.csv')
d2 = pd.read_csv(PROCESSED / 'yilin_d2_retail_clothing_clean.csv')

combined = pd.DataFrame({
    'gender_label': pd.concat([
        d1['Product_Gender'],
        d2['Gender'].map({'Female': 'Women', 'Male': 'Men'})
    ], ignore_index=True),
    'unit_price': pd.concat([
        d1['Avg_Price'],
        d2['Price per Unit']
    ], ignore_index=True),
    'source': (['D1_ProductLabel'] * len(d1)) + (['D2_BuyerGender'] * len(d2))
})

print('D1 shape:', d1.shape)
print('D2 shape:', d2.shape)
print('Combined shape:', combined.shape)



## Results

### Exploratory Data Analysis

We focused the final report on the analyses that most directly answer our research question. Rather than including every intermediate EDA step from the checkpoint, we selected the clearest comparisons: a direct product-label comparison in Dataset 1, a second category check using outerwear, a broader seasonal look within Dataset 1, a buyer-gender analysis in Dataset 2, and a final cross-dataset comparison.



### Analysis 1 — T-Shirts & Shirts: Direct Product-Label Price Comparison

This is our cleanest “apples-to-apples” test in Dataset 1. We isolate the **T-Shirts & Shirts** category and compare average prices between products labeled for men and products labeled for women.


In [ ]:

tshirts = d1[d1['Clothing_Type'] == 'T-Shirts & Shirts']    .query("Product_Gender in ['Men', 'Women']")    .copy()

summary_t = tshirts.groupby('Product_Gender')['Avg_Price'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std'
).round(2)
print(summary_t)

women_t = tshirts.loc[tshirts['Product_Gender'] == 'Women', 'Avg_Price']
men_t = tshirts.loc[tshirts['Product_Gender'] == 'Men', 'Avg_Price']
u_t, p_t = mannwhitneyu(women_t, men_t, alternative='greater')
print("Mann-Whitney U test (Women > Men): U={u_t:.0f}, p={p_t:.4f}")

plt.figure(figsize=(7, 4))
sns.barplot(data=tshirts, x='Product_Gender', y='Avg_Price', estimator='mean', errorbar=None,
            hue='Product_Gender', legend=False)
plt.title('Mean Price in T-Shirts & Shirts')
plt.xlabel('Product Gender')
plt.ylabel('Average Price (USD)')
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()



In this category, the pattern went against our original hypothesis. Men’s shirts had the higher average price, and the statistical test did not support the claim that women’s shirts were more expensive. That means our strongest direct category comparison did **not** provide evidence of a pink tax.



### Analysis 2 — Outerwear: Does the Pattern Hold in a Higher-Priced Category?

We repeated the comparison in **Outerwear** to see whether the result changes in a more expensive clothing category.


In [ ]:

outerwear = d1[d1['Clothing_Type'] == 'Outerwear']    .query("Product_Gender in ['Men', 'Women']")    .copy()

summary_o = outerwear.groupby('Product_Gender')['Avg_Price'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std'
).round(2)
print(summary_o)

women_o = outerwear.loc[outerwear['Product_Gender'] == 'Women', 'Avg_Price']
men_o = outerwear.loc[outerwear['Product_Gender'] == 'Men', 'Avg_Price']
t_stat, p_welch = stats.ttest_ind(women_o, men_o, equal_var=False)
u_o, p_uo = mannwhitneyu(women_o, men_o, alternative='greater')
print("Welch t-test: t={t_stat:.3f}, p={p_welch:.4f}")
print("Mann-Whitney U test (Women > Men): U={u_o:.0f}, p={p_uo:.4f}")

plt.figure(figsize=(7, 4))
sns.boxplot(data=outerwear, x='Product_Gender', y='Avg_Price', hue='Product_Gender', legend=False)
plt.title('Outerwear Price Distribution by Product Gender')
plt.xlabel('Product Gender')
plt.ylabel('Average Price (USD)')
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()



Outerwear showed the same general direction as the shirts analysis: men’s products were slightly more expensive on average. Even when we moved to a higher-priced category, we still did not find support for the idea that women’s products were systematically priced higher in Dataset 1.



### Analysis 3 — Dataset 1 Overall: Seasonal Price Patterns

After looking at specific categories, we zoomed out and asked whether the gender price gap in Dataset 1 changed over time. This helps us see whether any apparent pattern is stable across the year or whether it depends on seasonality.


In [ ]:

d1_season = d1.copy()
d1_season = d1_season.query("Product_Gender in ['Men', 'Women']")
d1_season['Season'] = np.where(d1_season['Month'] <= 6, 'Jan–Jun', 'Jul–Dec')

monthly = d1_season.groupby(['Month', 'Product_Gender'])['Avg_Price'].mean().reset_index()
print(monthly.pivot(index='Month', columns='Product_Gender', values='Avg_Price').round(2))

season_summary = d1_season.groupby(['Season', 'Product_Gender'])['Avg_Price'].agg(
    count='count', mean='mean', median='median'
).round(2)
print("Season summary:")
print(season_summary)

plt.figure(figsize=(10, 4))
for gender in ['Men', 'Women']:
    sub = monthly[monthly['Product_Gender'] == gender]
    plt.plot(sub['Month'], sub['Avg_Price'], marker='o', label=gender)
plt.xticks(range(1, 13))
plt.title('Monthly Mean Price by Product Gender (Dataset 1)')
plt.xlabel('Month')
plt.ylabel('Mean Price (USD)')
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.legend()
plt.tight_layout()
plt.show()



This broader view showed that the relationship was **not consistent across the year**. In the first half of the year, women’s items tended to have a higher average price, while in the second half, men’s items tended to be higher. That makes the overall story much more nuanced: instead of a stable pink tax pattern, Dataset 1 suggests that clothing prices may be shaped more by season and product mix.



### Analysis 4 — Dataset 2: Buyer-Gender Price Patterns

Dataset 2 cannot directly compare men’s and women’s products, because its gender variable refers to the **buyer** rather than the item. Still, it gives us a useful secondary perspective on whether price patterns differ by gender in another retail setting.


In [ ]:

d2_analysis = d2.copy()
d2_analysis['Age_Group'] = pd.cut(
    d2_analysis['Age'],
    bins=[17, 30, 45, 65],
    labels=['Young (18–30)', 'Middle (31–45)', 'Mature (46–64)']
)
d2_analysis['High_Value_Flag'] = (d2_analysis['Price per Unit'] >= 300).astype(int)

summary_d2 = d2_analysis.groupby('Gender')['Price per Unit'].agg(
    count='count', mean='mean', median='median', std='std'
).round(2)
print(summary_d2)

u_d2, p_d2 = mannwhitneyu(
    d2_analysis.loc[d2_analysis['Gender'] == 'Female', 'Price per Unit'],
    d2_analysis.loc[d2_analysis['Gender'] == 'Male', 'Price per Unit'],
    alternative='greater'
)
print("Mann-Whitney U test (Female > Male): U={u_d2:.0f}, p={p_d2:.4f}")

hv_age = d2_analysis.groupby(['Age_Group', 'Gender'])['High_Value_Flag'].mean().mul(100).round(1).reset_index()
print("High-value purchase rate (%):")
print(hv_age)

plt.figure(figsize=(8, 4))
sns.barplot(data=d2_analysis, x='Gender', y='Price per Unit', estimator='mean', errorbar=None,
            hue='Gender', legend=False)
plt.title('Mean Price per Unit by Buyer Gender (Dataset 2)')
plt.xlabel('Buyer Gender')
plt.ylabel('Mean Price per Unit (USD)')
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()



Dataset 2 pointed in a different direction: female buyers had a somewhat higher average unit price than male buyers. However, the difference was not strong enough to count as convincing statistical evidence, and this dataset is harder to interpret because buyer gender is only a proxy for product gender. We also found that the strongest female-higher pattern appeared among younger buyers, especially in high-value purchases, but that result was still not enough to support a broad claim on its own.



### Analysis 5 — Cross-Dataset Comparison

Finally, we aligned both datasets into a shared format so we could compare the overall direction of the price gap side by side.


In [ ]:

summary_combined = combined.groupby(['source', 'gender_label'])['unit_price'].agg(
    count='count', mean='mean', median='median', std='std'
).round(2).reset_index()
print(summary_combined)

results = []
for src in ['D1_ProductLabel', 'D2_BuyerGender']:
    women = combined[(combined['source'] == src) & (combined['gender_label'] == 'Women')]['unit_price']
    men = combined[(combined['source'] == src) & (combined['gender_label'] == 'Men')]['unit_price']
    u, p = mannwhitneyu(women, men, alternative='greater')
    results.append({
        'Source': src,
        'Mean Women': round(women.mean(), 2),
        'Mean Men': round(men.mean(), 2),
        'p-value': round(p, 4)
    })
print("Cross-dataset test results:")
print(pd.DataFrame(results))

plot_df = summary_combined.copy()
plot_df['source'] = plot_df['source'].map({
    'D1_ProductLabel': 'D1: Product Label',
    'D2_BuyerGender': 'D2: Buyer Gender'
})

plt.figure(figsize=(8, 4))
sns.barplot(data=plot_df, x='source', y='mean', hue='gender_label')
plt.title('Mean Unit Price by Gender Across Both Datasets')
plt.xlabel('Data Source')
plt.ylabel('Mean Unit Price (USD)')
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()



This comparison made the main takeaway very clear: the two datasets do **not** tell the same story. Dataset 1 tends to show men’s products costing more in the categories we examined, while Dataset 2 shows women’s purchases averaging a bit higher. Since the direction changes depending on the data source and how gender is measured, we do not see robust evidence for a single, stable pink tax pattern in these clothing datasets.



## Ethics

There are a few important limitations and ethical concerns in this project. First, gender is simplified into binary labels in both datasets, which leaves out nonbinary identities and reduces a more complex social category into a narrow variable. Second, Dataset 2 uses **buyer gender** rather than **product gender**, so it should not be interpreted as direct evidence that women’s clothing costs more. Third, our keyword-based labeling in Dataset 1 is practical, but it may misclassify some products or miss subtle product differences that affect price.

We also want to be careful not to overstate the meaning of observed price gaps. Even when one group has a higher average price, that does not automatically prove discrimination; differences in material, style, category mix, or seasonal inventory can also matter. For that reason, we treat this project as an exploratory analysis rather than a definitive causal claim about gender-based pricing.



## Discussion and Conclusion

We did this project to test whether a pink tax could be detected in clothing prices using real retail data. Our original expectation was that women’s clothing would show consistently higher prices. After comparing two different datasets, we found that the answer was much more mixed than expected.

In Dataset 1, which is the stronger dataset for this question because it measures **product label gender**, our category-level analyses did not support the hypothesis. In both **T-Shirts & Shirts** and **Outerwear**, men’s items were slightly more expensive on average. When we looked at Dataset 1 more broadly, we also found that the direction of the gap changed over the year, which suggests that seasonality and product mix matter a lot.

Dataset 2 told a somewhat different story. Female buyers showed a slightly higher average price per unit, and younger women seemed especially likely to appear in higher-value purchases. But because the gender variable refers to the customer rather than the clothing item, these results are harder to interpret as evidence of a pink tax. On top of that, the statistical evidence was not strong enough to support a clear conclusion on its own.

Overall, our final conclusion is that we did **not** find strong, consistent evidence of a pink tax in these clothing datasets. Instead, the project suggests that the observed price differences depend heavily on category, season, and especially on how gender is defined in the data. That is probably the most important takeaway from our work: when the measurement changes, the conclusion can change too.
